# Clase 04 -- Ciclo de vida de los datos (Parte II)

Practica de las etapas **03 acceso/seguridad**, **04 limpieza/analisis/
visualizacion** y **05 archivado/eliminacion**. Vamos a:

1. **Seguridad y privacidad (03)**: anonimizar datos personales (PII) con
   hashing (SHA-256) y enmascarado, para poder usarlos sin exponer identidades.
2. **EDA (04)**: describir, contar faltantes, detectar outliers con la regla
   del IQR y medir correlacion. Ademas, el **cuarteto de Anscombe** muestra por
   que SIEMPRE hay que visualizar.
3. **Archivado y eliminacion (05)**: atender una solicitud **ARCO de
   cancelacion** (borrar un registro) y archivar el resto comprimido con un
   JSON de metadatos (fecha, responsable, razon).

Uso:
    python practica.py
Genera datasets en datos/ (incluido el archivo comprimido) y graficas en figuras/.

In [1]:
# Imports (stdlib y externos)
import gzip
import hashlib
import json
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")  # backend sin ventana: guardamos las figuras a archivo
import matplotlib.pyplot as plt

In [2]:
# Carpetas de salida. Como script usamos __file__; como notebook, el cwd.
try:
    BASE = Path(__file__).resolve().parent
except NameError:
    BASE = Path.cwd()
DATOS = BASE / "datos"
FIGURAS = BASE / "figuras"
DATOS.mkdir(exist_ok=True)
FIGURAS.mkdir(exist_ok=True)

rng = np.random.default_rng(11)

## 1. Seguridad y privacidad (etapa 03): anonimizar PII

Antes de compartir datos, hay que proteger la informacion personal (PII).
Creamos una tabla de clientes con nombre, correo, edad e ingreso, y la
anonimizamos: el nombre+correo se convierten en un seudonimo con SHA-256 y el
correo se enmascara. Asi se puede analizar sin revelar identidades.

In [3]:
# Tabla de clientes con datos personales (PII)
clientes = pd.DataFrame({
    "id": [1, 2, 3, 4, 5, 6, 7, 8],
    "nombre": ["Ana Ruiz", "Luis Gil", "Mara Paz", "Beto Lima",
               "Eva Soto", "Noe Diaz", "Ivan Mora", "Sara Vega"],
    "email": ["ana@mail.com", "luis@mail.com", "mara@mail.com", "beto@mail.com",
              "eva@mail.com", "noe@mail.com", "ivan@mail.com", "sara@mail.com"],
    "edad": [34, 29, 41, 23, 52, 38, 45, None],          # un faltante
    "ingreso": [12000, 18000, 15000, 9000, 25000, 16000, 200000, 14000],  # un outlier
})
print("Clientes con PII (NO se debe compartir asi):")
print(clientes[["id", "nombre", "email"]])

Clientes con PII (NO se debe compartir asi):
   id     nombre          email
0   1   Ana Ruiz   ana@mail.com
1   2   Luis Gil  luis@mail.com
2   3   Mara Paz  mara@mail.com
3   4  Beto Lima  beto@mail.com
4   5   Eva Soto   eva@mail.com
5   6   Noe Diaz   noe@mail.com
6   7  Ivan Mora  ivan@mail.com
7   8  Sara Vega  sara@mail.com


In [4]:
# Funciones de anonimizacion: seudonimo (hash) y enmascarado del correo
def seudonimo(texto: str) -> str:
    # SHA-256 produce un identificador estable y no reversible; tomamos 10 chars
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()[:10]

def enmascarar_email(email: str) -> str:
    # deja la primera letra y oculta el resto del usuario: a***@mail.com
    usuario, dominio = email.split("@")
    return usuario[0] + "***@" + dominio

anon = clientes.copy()
anon["seudonimo"] = (anon["nombre"] + anon["email"]).map(seudonimo)
anon["email"] = anon["email"].map(enmascarar_email)
anon = anon.drop(columns=["nombre"])    # quitamos el nombre real
anon.to_csv(DATOS / "clientes_anon.csv", index=False)
print("Datos anonimizados (listos para analizar/compartir):")
print(anon[["id", "seudonimo", "email", "edad", "ingreso"]])

Datos anonimizados (listos para analizar/compartir):
   id   seudonimo          email  edad  ingreso
0   1  8d871fa84b  a***@mail.com  34.0    12000
1   2  663d97c1fc  l***@mail.com  29.0    18000
2   3  0383d6d248  m***@mail.com  41.0    15000
3   4  8da0bcfd62  b***@mail.com  23.0     9000
4   5  19a77f0c51  e***@mail.com  52.0    25000
5   6  702e8c6c0f  n***@mail.com  38.0    16000
6   7  88c6f032c5  i***@mail.com  45.0   200000
7   8  5d0ab54419  s***@mail.com   NaN    14000


## 2. EDA (etapa 04): describir, faltantes, outliers y correlacion

Con los datos ya anonimizados hacemos un analisis exploratorio basico.

In [5]:
# Resumen descriptivo de las variables numericas
print("Resumen descriptivo:")
print(anon[["edad", "ingreso"]].describe())
print("\nValores faltantes por columna:")
print(anon.isna().sum())

Resumen descriptivo:
            edad        ingreso
count   7.000000       8.000000
mean   37.428571   38625.000000
std     9.778499   65373.514733
min    23.000000    9000.000000
25%    31.500000   13500.000000
50%    38.000000   15500.000000
75%    43.000000   19750.000000
max    52.000000  200000.000000

Valores faltantes por columna:
id           0
email        0
edad         1
ingreso      0
seudonimo    0
dtype: int64


In [6]:
# Deteccion de outliers en el ingreso con la regla 1.5 x IQR
q1 = anon["ingreso"].quantile(0.25)
q3 = anon["ingreso"].quantile(0.75)
iqr = q3 - q1
bajo, alto = q1 - 1.5 * iqr, q3 + 1.5 * iqr
atipicos = anon[(anon["ingreso"] < bajo) | (anon["ingreso"] > alto)]
print(f"Limites normales de ingreso: [{bajo:,.0f} , {alto:,.0f}]")
print("Outliers detectados:")
print(atipicos[["id", "seudonimo", "ingreso"]])

Limites normales de ingreso: [4,125 , 29,125]
Outliers detectados:
   id   seudonimo  ingreso
6   7  88c6f032c5   200000


In [7]:
# Correlacion entre edad e ingreso (ignorando el faltante)
correlacion = anon[["edad", "ingreso"]].corr()
print("Matriz de correlacion edad-ingreso:")
print(correlacion.round(3))

Matriz de correlacion edad-ingreso:
          edad  ingreso
edad     1.000    0.395
ingreso  0.395    1.000


### ?`Por que SIEMPRE visualizar? El cuarteto de Anscombe

Los cuatro conjuntos de Anscombe tienen casi las mismas estadisticas (media,
desviacion, correlacion y recta de regresion) pero formas muy distintas. Lo
comprobamos numericamente y luego lo graficamos.

In [8]:
# Cuarteto de Anscombe (valores clasicos)
x123 = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
anscombe = pd.DataFrame({
    "x1": x123, "y1": [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68],
    "x2": x123, "y2": [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74],
    "x3": x123, "y3": [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73],
    "x4": [8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],
    "y4": [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89],
})
for i in range(1, 5):
    x, y = anscombe[f"x{i}"], anscombe[f"y{i}"]
    print(f"Set {i}: media_y={y.mean():.2f}, std_y={y.std(ddof=1):.2f}, "
          f"corr={x.corr(y):.3f}")

Set 1: media_y=7.50, std_y=2.03, corr=0.816
Set 2: media_y=7.50, std_y=2.03, corr=0.816
Set 3: media_y=7.50, std_y=2.03, corr=0.816
Set 4: media_y=7.50, std_y=2.03, corr=0.817


In [9]:
# Graficamos los cuatro conjuntos: misma estadistica, formas distintas
fig, axes = plt.subplots(2, 2, figsize=(7, 5.5))
colores = ["#0094A2", "#FE730C", "#FEB200", "#532587"]
for i, ax in enumerate(axes.flat, start=1):
    x, y = anscombe[f"x{i}"], anscombe[f"y{i}"]
    ax.scatter(x, y, color=colores[i - 1])
    # recta de regresion (identica en los cuatro): y = 3 + 0.5 x
    xs = np.array([3, 20])
    ax.plot(xs, 3 + 0.5 * xs, color="gray", linewidth=1, linestyle="--")
    ax.set_title(f"Set {i}")
    ax.set_xlim(2, 20)
    ax.set_ylim(2, 14)
fig.suptitle("Cuarteto de Anscombe: misma estadistica, formas distintas")
fig.tight_layout()
fig.savefig(FIGURAS / "anscombe.png", dpi=130)
plt.close(fig)
print("Figura guardada: figuras/anscombe.png")

Figura guardada: figuras/anscombe.png


## 3. Archivado y eliminacion (etapa 05)

### 3a. Solicitud ARCO de cancelacion

Un cliente ejerce su derecho de **cancelacion** (eliminacion). Borramos su
registro de la base activa, dejando constancia de la operacion.

In [10]:
# El cliente con id=4 solicita la eliminacion de sus datos
id_baja = 4
antes = len(anon)
activos = anon[anon["id"] != id_baja].copy()
print(f"Registros antes: {antes}; despues de la cancelacion: {len(activos)}")
print("?`Sigue el id=4 en la base activa?", bool((activos['id'] == id_baja).any()))
activos.to_csv(DATOS / "clientes_activos.csv", index=False)

Registros antes: 8; despues de la cancelacion: 7
?`Sigue el id=4 en la base activa? False


### 3b. Archivar el resto comprimido y con metadatos

Los registros que se conservan se archivan **comprimidos** (ahorro de espacio)
y acompanados de un JSON de **metadatos** que documenta la operacion: cuando,
quien y por que (clave para auditoria y cumplimiento).

In [11]:
# Guardamos el archivo comprimido (almacenamiento "frio")
ruta_archivo = DATOS / "clientes_archivo.csv.gz"
with gzip.open(ruta_archivo, "wt") as f:
    activos.to_csv(f, index=False)

# Metadatos de la operacion de archivado (para auditoria)
contenido = ruta_archivo.read_bytes()
metadatos = {
    "fecha": date.today().isoformat(),
    "responsable": "equipo_datos",
    "razon": "archivado de clientes activos; baja ARCO del id 4",
    "n_registros": int(len(activos)),
    "archivo": ruta_archivo.name,
    "sha256": hashlib.sha256(contenido).hexdigest(),  # huella de integridad
}
(DATOS / "clientes_archivo.meta.json").write_text(
    json.dumps(metadatos, ensure_ascii=True, indent=2))
print("Metadatos del archivado:")
print(json.dumps(metadatos, ensure_ascii=True, indent=2))

Metadatos del archivado:
{
  "fecha": "2026-06-29",
  "responsable": "equipo_datos",
  "razon": "archivado de clientes activos; baja ARCO del id 4",
  "n_registros": 7,
  "archivo": "clientes_archivo.csv.gz",
  "sha256": "61d528352f952964485ba3823ceacd04e5604a1db0c1df32d9fb535c893ca490"
}


## Conclusiones

- **Seguridad (03)**: anonimizar PII (seudonimos con hash y enmascarado)
  permite analizar los datos sin exponer identidades.
- **EDA (04)**: describir, contar faltantes, detectar outliers con el IQR y
  medir correlaciones son los primeros pasos de todo analisis.
- **Visualizar siempre**: el cuarteto de Anscombe tiene la misma estadistica
  pero formas muy distintas; sin graficar, no se nota.
- **Archivado/eliminacion (05)**: atender solicitudes ARCO de cancelacion y
  archivar comprimido con metadatos (fecha, responsable, razon, huella SHA-256)
  deja una operacion trazable y auditable.